# REMOVE ROW

In [3]:
from pathlib import Path
import pyarrow as pa
import pyarrow.parquet as pq
from tqdm.auto import tqdm
import json

def print_and_drop_row_by_index_streaming(
    in_path: str | Path,
    out_path: str | Path,
    drop_index: int,
    *,
    batch_rows: int = 10_000,
    print_max_chars: int = 2000,  # prevent dumping huge article_text to output
) -> None:
    in_path = Path(in_path).resolve()
    out_path = Path(out_path).resolve()
    out_path.parent.mkdir(parents=True, exist_ok=True)

    pf = pq.ParquetFile(str(in_path))
    total_rows = pf.metadata.num_rows

    if drop_index < 0 or drop_index >= total_rows:
        raise IndexError(f"drop_index out of range: {drop_index} (0..{total_rows-1})")

    writer = None
    seen = 0
    printed = False

    def _truncate(v):
        if isinstance(v, str) and len(v) > print_max_chars:
            return v[:print_max_chars] + f"... [truncated {len(v)-print_max_chars} chars]"
        return v

    with tqdm(total=total_rows, desc=f"drop row {drop_index}", unit="rows") as pbar:
        for batch in pf.iter_batches(batch_size=batch_rows):
            table = pa.Table.from_batches([batch])
            n = table.num_rows

            # global range: [seen, seen+n)
            if (not printed) and (seen <= drop_index < seen + n):
                local = drop_index - seen
                one = table.slice(local, 1).to_pydict()
                row = {k: _truncate(v[0]) for k, v in one.items()}
                print("Row to delete (index={}):".format(drop_index))
                print(json.dumps(row, indent=2, ensure_ascii=False))
                printed = True

                # remove that row from this batch
                keep_mask = [True] * n
                keep_mask[local] = False
                table = table.filter(pa.array(keep_mask))

            # write batch
            if writer is None:
                writer = pq.ParquetWriter(str(out_path), table.schema)
            writer.write_table(table)

            seen += n
            pbar.update(n)

    if writer is not None:
        writer.close()

    print("Wrote:", out_path)


In [9]:
IN_FILE = r"C:\Users\insoo\Documents\Western_WAI\explainable-misinfo-ai\data\processed\_coaid_split_tmp\coaid_false_full.parquet"
OUT_FILE = r"C:\Users\insoo\Documents\Western_WAI\explainable-misinfo-ai\data\processed\_coaid_split_tmp\coaid_false_full_cleaned.parquet"
DROP_LINE = 320  # <-- your “line” (0-based row index)

print_and_drop_row_by_index_streaming(
    IN_FILE, OUT_FILE, 
    DROP_LINE,
    batch_rows=5000,
    print_max_chars=2000,
)

drop row 320:   0%|          | 0/513 [00:00<?, ?rows/s]

Row to delete (index=320):
{
  "dataset": "coaid",
  "id": "coaid_07-01-2020_news_false_758",
  "claim_text": "\"guess what children are not at risk unless they have an underlying medical issue.\"",
  "article_text": "k e 2mdia mdhd d d hdlr vide mainconcept video media handler minf vmhd 3hdlr alis alias data handler dinf avc coding 0avcc m ) gm ) r o 5 . ! h 5 stts stss / f t , c z q ) w n t k c5 cl cc cz c c c c c d d d2 di d dw d d d d d e e e/ ef e et e e e e e e f f f ft fk f f f f f f g g# g: gq gh g g g g g g h h h7 hn he h h h h h h i i i4 ik ib iy i i i i i j j j1 jh j_ jv j j j j j k k k. ke k ks k k k k k k l l lb ly lp l l l l l l m m( m? mv mm m m m m m m n n n< ns nj n n n n n n o o\" o9 op og o o o o o o p p p6 pm pd p p p p p p q q q3 qj qa qx q q q q q r r r0 rg r ru r r r r r r s s- sd s sr s s s s s s t t ta tx to t t t t t t u u' u> uu ul u u u u u u v x5 xl xc xz x x x x x y y y2 yi y yw y y y y y z z z/ zf z zt z z z z z z , c z q ) w n t k 4 k b y a a a1 ah a_ av

# PRINT NUM OF ROW

In [7]:
import pandas as pd
from pathlib import Path
import pyarrow.parquet as pq

def print_parquet_num_rows(path: str | Path) -> int:
    """
    Fast row count (reads parquet metadata only).
    """
    path = Path(path).resolve()
    pf = pq.ParquetFile(str(path))
    n = pf.metadata.num_rows
    print(f"{path.name}: {n:,} rows")
    return n


print_parquet_num_rows("C:/Users/insoo/Documents/Western_WAI/explainable-misinfo-ai/data/processed/_coaid_split_tmp/coaid_false_full.parquet")



coaid_false_full.parquet: 513 rows


513

# PRINT HUGE ONES

In [10]:
import pyarrow as pa
import pyarrow.parquet as pq
from tqdm.auto import tqdm

def find_huge_article_rows(path, *, top_k=10, batch_rows=256):
    pf = pq.ParquetFile(path)
    total = pf.metadata.num_rows

    best = []  # list of (len, global_idx)
    seen = 0

    with tqdm(total=total, desc="scan article_text lengths", unit="rows") as pbar:
        for batch in pf.iter_batches(batch_size=batch_rows, columns=["article_text"]):
            tbl = pa.Table.from_batches([batch])
            arts = tbl.column("article_text").to_pylist()
            for i, a in enumerate(arts):
                L = len(str(a)) if a is not None else 0
                gi = seen + i
                best.append((L, gi))
            seen += len(arts)
            pbar.update(len(arts))

    best.sort(reverse=True)
    print("Top huge rows (len, index):")
    for L, gi in best[:top_k]:
        print(f"  idx={gi:>6}  len={L:,}")
    return best[:top_k]

# Example:
find_huge_article_rows("C:/Users/insoo/Documents/Western_WAI/explainable-misinfo-ai/data/processed/_coaid_split_tmp/coaid_true_full.parquet", top_k=5)


scan article_text lengths:   0%|          | 0/4755 [00:00<?, ?rows/s]

Top huge rows (len, index):
  idx=  1666  len=35,537
  idx=   603  len=21,726
  idx=   596  len=20,457
  idx=  1694  len=17,955
  idx=  1693  len=17,955


[(35537, 1666), (21726, 603), (20457, 596), (17955, 1694), (17955, 1693)]